# [V3] Huan luyen Faster R-CNN ResNet50-FPN - Zalo AI Traffic Sign 2020

Ban V3 nang cap tu ban goc. Day la notebook thay doi nhieu nhat trong 3 model, vi Faster R-CNN dung PyTorch thuan nen moi thu deu phai tu viet tay:

| Hang muc | Ban cu | Ban V3 | Ly do |
|---|---|---|---|
| Chia du lieu | `random_split` 90/10, **khong set seed** | **70/10/20 seed 42**, dung chung script voi 2 model kia | Ban cu khong tai lap duoc, va tap val cua no chong len tap test cua 2 model kia |
| So epoch | 15 (chay du roi dung) | **100 + Early Stopping `patience=15`** | De model tu quyet dinh diem hoi tu |
| Scheduler | `StepLR(step_size=10, gamma=0.1)` | **`CosineAnnealingLR`** | `StepLR` voi 100 epoch se lam LR teo ve gan 0 tu epoch 30, model dung hoc. Doi sang Cosine cho dong bo voi `cos_lr=True` cua 2 model kia |
| Chon Best Model | Theo `val_loss` | Theo **mAP@50-95** tren tap Val | Ultralytics chon best theo mAP, nen Faster R-CNN cung phai theo mAP thi 3 model moi cung mot tieu chi |
| Augmentation tap Val | Dung chung `get_transform()` voi tap train (**co Random Crop**) | Tap Val dung transform rieng, **khong augment** | Augment tap val lam diem danh gia nhieu loan |
| Nhat ky | Chi in ra man hinh | Xuat `faster_rcnn_training_history.json` | De ve Learning Curve |

**Moi truong:** Kaggle Notebook, GPU P100 hoac T4. **Dataset can Add:** `phhasian0710/za-traffic-2020`.

> ### Nguyen tac vang cua phien ban V3
>
> Toan bo du lieu duoc chia lai theo ty le **70% Train / 10% Validation / 20% Hold-out Test** bang `split_dataset.py` voi `random_seed=42`.
>
> - Thu muc `holdout_test/` **khong duoc khai bao trong `data.yaml`** va tuyet doi khong dung o bat ky buoc nao trong notebook nay.
> - Ca 3 model deu goi cung mot script chia, cung mot seed, nen chac chan dung chung mot tap Test.
> - Chi mo tap Hold-out ra dung mot lan duy nhat o notebook `evaluate_3_models.ipynb`.

## Cell 1: Cai thu vien va kiem tra GPU

Can `torchmetrics` de tinh mAP sau moi epoch. Day la diem moi so voi ban cu (ban cu chi theo doi `val_loss`), bat buoc phai co thi moi ghi duoc `mAP_50` va `mAP_50_95` vao file nhat ky.

**Luu y chon GPU:** phai chon **T4 x2**, khong duoc chon **P100**. Ban PyTorch trong moi truong Kaggle hien tai duoc bien dich cho kien truc tu sm_70 tro len, trong khi P100 la kien truc Pascal sm_60 nen khong co ma may de chay, se bao loi `CUDA error: no kernel image is available for execution on the device`. T4 la sm_75 nen chay binh thuong.

Doan kiem tra ben duoi chay thu mot phep tinh nho tren GPU. Neu chon nham GPU thi no bao loi ngay o cell dau, khoi phai doi den luc train moi biet.

In [ ]:
!pip install -q torchmetrics

import torch

print(f"PyTorch       : {torch.__version__}")
print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"Compute cap   : sm_{''.join(map(str, torch.cuda.get_device_capability(0)))}")
print(f"PyTorch ho tro: {torch.cuda.get_arch_list()}")

# Chay thu mot phep tinh nho tren GPU. Neu ban PyTorch khong co ma may cho kien truc
# GPU dang chon (vi du P100 la sm_60) thi dong nay nem loi ngay, thay vi de notebook
# chay het phan chia du lieu roi moi chet o giua vong lap train.
try:
    thu = torch.randn(100, 100, device='cuda')
    _ = (thu @ thu).sum().item()
    print("\nGPU chay duoc binh thuong.")
except Exception as loi:
    raise RuntimeError(
        f"GPU dang chon khong tuong thich voi ban PyTorch cua moi truong ({loi}).\n"
        "Cach xu ly: o thanh ben phai, muc Accelerator doi tu 'GPU P100' sang 'GPU T4 x2' "
        "roi chay lai. P100 la kien truc Pascal sm_60, da bi cac ban PyTorch dung CUDA 12.8 "
        "tro len bo ho tro."
    )

## Cell 2: Lay script chia du lieu va chay

In [ ]:
import os
import urllib.request

# Keo file split_dataset.py tu GitHub ve de dung chung mot phep chia voi 2 model kia.
URL_SCRIPT = ('https://raw.githubusercontent.com/vtdung23/Object-Detection-Application/main/Traffic-Sign-Detection-ZaloAI/data_preparation/split_dataset.py')

if not os.path.exists('split_dataset.py'):
    try:
        urllib.request.urlretrieve(URL_SCRIPT, 'split_dataset.py')
        print("Da tai split_dataset.py tu GitHub ve.")
    except Exception as loi:
        raise FileNotFoundError(
            f"Khong tai duoc split_dataset.py ({loi}).\n"
            "Cach khac: mo file split_dataset.py trong repo, copy noi dung roi tao thu cong "
            "file cung ten o thu muc lam viec hien tai."
        )
else:
    print("File split_dataset.py da co san.")

In [ ]:
!python split_dataset.py --output-root /kaggle/working/data_v3

THU_MUC_DATA = '/kaggle/working/data_v3/dataset_train_val'
JSON_TRAIN = f'{THU_MUC_DATA}/train_annotations.json'
JSON_VAL = f'{THU_MUC_DATA}/val_annotations.json'
ANH_TRAIN = f'{THU_MUC_DATA}/train/images'
ANH_VAL = f'{THU_MUC_DATA}/val/images'

print(f"JSON train: {JSON_TRAIN}")
print(f"JSON val  : {JSON_VAL}")

## Cell 3: Lop Dataset doc COCO JSON

Faster R-CNN doc thang COCO JSON chu khong doc file `.txt` cua YOLO. Vi vay `split_dataset.py` da xuat san `train_annotations.json` va `val_annotations.json` — hai file nay chua dung nhung anh ma YOLOv8 va RT-DETR duoc hoc, khong lech mot tam nao.

Bounding box phai doi tu he COCO `[x, y, w, h]` sang Pascal VOC `[xmin, ymin, xmax, ymax]` vi torchvision chi hieu he sau.

In [ ]:
import os
import json

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2


def get_transform_train():
    """Augmentation cho tap Train: cat khuon 512x512 an toan bounding box."""
    return A.Compose([
        # Anh goc Zalo cao 626px nen 512 la muc Power of 2 an toan nhat
        A.RandomSizedBBoxSafeCrop(width=512, height=512, erosion_rate=0.0, p=0.3),
        A.Normalize(),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.5))


def get_transform_val():
    """Tap Val KHONG duoc augment, chi chuan hoa. Ban cu dung chung transform
    voi tap train nen diem danh gia bi nhieu loan boi Random Crop."""
    return A.Compose([
        A.Normalize(),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.5))


class ZaloTrafficDataset(Dataset):
    """Doc du lieu tu file COCO JSON do split_dataset.py sinh ra."""

    def __init__(self, thu_muc_anh, duong_dan_json, transform=None):
        self.thu_muc_anh = thu_muc_anh
        self.transform = transform

        with open(duong_dan_json, 'r', encoding='utf-8') as f:
            self.coco_data = json.load(f)

        # Gom bbox theo image_id de truy xuat O(1)
        self.bbox_theo_anh = {}
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.bbox_theo_anh:
                self.bbox_theo_anh[img_id] = []
            self.bbox_theo_anh[img_id].append(ann)

        self.images = self.coco_data['images']

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        thong_tin = self.images[idx]
        img_id = thong_tin['id']
        duong_dan = os.path.join(self.thu_muc_anh, thong_tin['file_name'])

        image = cv2.imread(duong_dan)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        boxes = []
        labels = []
        for ann in self.bbox_theo_anh.get(img_id, []):
            x, y, w, h = ann['bbox']
            boxes.append([x, y, x + w, y + h])   # COCO -> Pascal VOC
            labels.append(ann['category_id'])    # giu nguyen 1..7, so 0 danh cho background

        if len(boxes) == 0:
            boxes = np.zeros((0, 4), dtype=np.float32)
            labels = np.zeros((0,), dtype=np.int64)

        if self.transform:
            ket_qua = self.transform(image=image, bboxes=boxes, labels=labels)
            image = ket_qua['image']
            boxes = ket_qua['bboxes']
            labels = ket_qua['labels']

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        # Sau khi crop co the khong con box nao, phai ep lai dung shape (0, 4)
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([img_id]),
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = ZaloTrafficDataset(ANH_TRAIN, JSON_TRAIN, transform=get_transform_train())
val_dataset = ZaloTrafficDataset(ANH_VAL, JSON_VAL, transform=get_transform_val())

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn
)

print(f"Train: {len(train_dataset)} anh | Val: {len(val_dataset)} anh")

## Cell 4: Tinh Anchor Box bang K-Means

Diem khac quan trong so voi ban cu: K-Means chi duoc chay tren **tap Train**, khong duoc dung toan bo dataset.

Ly do: kich thuoc anchor la mot tham so ma model hoc duoc tu du lieu. Neu chay K-Means tren ca 4500 anh thi thong tin ve kich thuoc bien bao trong tap Test da ro ri vao thiet ke mang — dung nghia Data Leakage, du rat nhe. Chay rieng tren tap Train la cach lam sach se ve mat hoc thuat.

In [ ]:
from sklearn.cluster import KMeans

# CHI lay bbox cua tap Train de tranh ro ri thong tin tu tap Test
danh_sach_kich_thuoc = []
for ann in train_dataset.coco_data['annotations']:
    danh_sach_kich_thuoc.append([ann['bbox'][2], ann['bbox'][3]])

print(f"Dang chay K-Means tren {len(danh_sach_kich_thuoc)} bounding box cua tap Train...")

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(danh_sach_kich_thuoc)
centers = np.sort(kmeans.cluster_centers_, axis=0)
anchor_sizes_kmeans = tuple(int(center[0]) for center in centers)

print(f"5 kich thuoc Anchor thu duoc: {anchor_sizes_kmeans}")

# FPN can 5 tuple rieng biet cho 5 tang, ep ty le 1:1 vi bien bao co tinh doi xung cao
ANCHOR_SIZES = tuple((size,) for size in anchor_sizes_kmeans)
ASPECT_RATIOS = ((1.0,),) * len(ANCHOR_SIZES)

## Cell 5: Dung mo hinh Faster R-CNN

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Thiet bi: {DEVICE}")

NUM_CLASSES = 8  # 7 loai bien bao + 1 class background bat buoc cua PyTorch


def get_model(num_classes, anchor_sizes, aspect_ratios):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')

    # Ep bo Anchor K-Means vao mang RPN thay cho bo mac dinh cua COCO
    model.rpn.anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)

    # Thay lop phan loai 91 class cua COCO bang 8 class cua bai toan nay
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model


model = get_model(NUM_CLASSES, ANCHOR_SIZES, ASPECT_RATIOS)
model.to(DEVICE)
print("Da dung xong mo hinh Faster R-CNN voi Anchor K-Means.")

## Cell 6: Ham do mAP tren tap Validation

Ban cu chi do `val_loss`. Ban V3 phai do them mAP sau moi epoch vi hai ly do: file nhat ky yeu cau co truong `mAP_50` va `mAP_50_95`, va Early Stopping se dua vao mAP de quyet dinh diem dung (giong cach Ultralytics lam voi 2 model kia).

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision


@torch.no_grad()
def do_map_tren_val(model, data_loader):
    """Chay che do eval de lay bounding box du doan, roi tinh mAP bang torchmetrics."""
    model.eval()
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')

    for images, targets in data_loader:
        images = [img.to(DEVICE) for img in images]
        du_doan = model(images)

        # torchmetrics can tensor nam tren CPU
        du_doan_cpu = [{k: v.detach().cpu() for k, v in p.items()} for p in du_doan]
        that_cpu = [{'boxes': t['boxes'], 'labels': t['labels']} for t in targets]

        metric.update(du_doan_cpu, that_cpu)

    ket_qua = metric.compute()
    return float(ket_qua['map_50']), float(ket_qua['map'])


@torch.no_grad()
def do_val_loss(model, data_loader):
    """Do Validation Loss.

    Meo quen thuoc: torchvision chi tra ve dict loss khi model o che do train(),
    con eval() thi tra ve bounding box. Nen phai giu model.train() nhung boc
    trong no_grad() de chan gradient - model khong hoc lom du lieu val.
    """
    model.train()
    tong_loss = 0.0
    so_batch = 0

    for images, targets in data_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        tong_loss += sum(loss for loss in loss_dict.values()).item()
        so_batch += 1

    return tong_loss / max(so_batch, 1)

## Cell 7: Vong lap huan luyen (100 epochs + Early Stopping + ghi JSON)

Ba diem can luu y trong vong lap nay:

1. **Early Stopping thu cong.** Ultralytics co san `patience`, con PyTorch thuan thi phai tu dem. Bien `so_epoch_khong_cai_thien` tang len moi khi mAP@50-95 khong pha duoc ky luc cu; cham 15 thi `break`.
2. **File JSON duoc ghi lai sau MOI epoch**, khong doi den luc train xong. Kaggle rat hay ngat phien giua chung, ghi lien tuc thi du co bi ngat van con nhat ky cua nhung epoch da chay.
3. **Scheduler doi sang Cosine Annealing.** Ban cu dung `StepLR(step_size=10, gamma=0.1)` — voi 15 epoch thi khong sao, nhung keo len 100 epoch thi Learning Rate se bi chia 10 tong cong 10 lan, tuc la teo con $0{,}005 \times 10^{-10}$. Model se dung hoc hoan toan tu khoang epoch 30. Cosine Annealing giam muot theo duong cong sin va dong bo voi `cos_lr=True` cua YOLOv8 va RT-DETR.

In [ ]:
import json
import time

from tqdm import tqdm

SO_EPOCHS = 100
PATIENCE = 15

THU_MUC_LUU = '/kaggle/working/faster_rcnn_v3'
os.makedirs(THU_MUC_LUU, exist_ok=True)

DUONG_DAN_BEST = os.path.join(THU_MUC_LUU, 'faster_rcnn_best.pth')
DUONG_DAN_JSON = '/kaggle/working/faster_rcnn_training_history.json'

# Giu nguyen SGD + Momentum: voi ResNet, SGD hoi tu on dinh hon ho Adam
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# [V3] Doi tu StepLR sang Cosine Annealing cho dong bo voi 2 model kia
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SO_EPOCHS)

lich_su = []
map_tot_nhat = -1.0
so_epoch_khong_cai_thien = 0
epoch_tot_nhat = 0


def ghi_nhat_ky():
    """Ghi file JSON lich su. Goi sau moi epoch de chong mat du lieu khi Kaggle ngat phien."""
    ket_qua = {
        'model': 'Faster R-CNN ResNet50-FPN',
        'epochs_du_kien': SO_EPOCHS,
        'epochs_thuc_te': len(lich_su),
        'early_stopping_patience': PATIENCE,
        'epoch_tot_nhat': epoch_tot_nhat,
        'mAP_50_95_tot_nhat': round(map_tot_nhat, 6),
        'history': lich_su,
    }
    with open(DUONG_DAN_JSON, 'w', encoding='utf-8') as f:
        json.dump(ket_qua, f, ensure_ascii=False, indent=2)
    return ket_qua


print(f"Bat dau huan luyen: toi da {SO_EPOCHS} epochs, Early Stopping patience={PATIENCE}\n")

for epoch in range(1, SO_EPOCHS + 1):
    thoi_diem_bat_dau = time.time()

    # ---------- PHA 1: TRAINING ----------
    model.train()
    tong_loss_train = 0.0

    thanh_tien_trinh = tqdm(train_loader, desc=f"Epoch {epoch}/{SO_EPOCHS} [Train]")
    for images, targets in thanh_tien_trinh:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        tong_loss_train += losses.item()
        thanh_tien_trinh.set_postfix(loss=losses.item())

    loss_train_tb = tong_loss_train / len(train_loader)

    # ---------- PHA 2: DO VAL LOSS VA mAP ----------
    loss_val_tb = do_val_loss(model, val_loader)
    map_50, map_50_95 = do_map_tren_val(model, val_loader)

    lich_su.append({
        'epoch_id': epoch,
        'train_loss': round(loss_train_tb, 6),
        'val_loss': round(loss_val_tb, 6),
        'mAP_50': round(map_50, 6),
        'mAP_50_95': round(map_50_95, 6),
    })

    thoi_gian = time.time() - thoi_diem_bat_dau
    print(f"Epoch {epoch}/{SO_EPOCHS} | Train Loss: {loss_train_tb:.4f} | "
          f"Val Loss: {loss_val_tb:.4f} | mAP@50: {map_50:.4f} | "
          f"mAP@50-95: {map_50_95:.4f} | {thoi_gian:.0f}s")

    # ---------- PHA 3: LUU BEST MODEL & DEM EARLY STOPPING ----------
    if map_50_95 > map_tot_nhat:
        map_tot_nhat = map_50_95
        epoch_tot_nhat = epoch
        so_epoch_khong_cai_thien = 0

        # Luu kem Anchor Box vao checkpoint. Neu chi luu state_dict thi luc load lai
        # de test se phai chay lai K-Means de doan anchor - rat de sai lech.
        torch.save({
            'model_state_dict': model.state_dict(),
            'anchor_sizes': ANCHOR_SIZES,
            'aspect_ratios': ASPECT_RATIOS,
            'num_classes': NUM_CLASSES,
            'epoch': epoch,
            'mAP_50_95': map_50_95,
        }, DUONG_DAN_BEST)
        print(f"  >> Ky luc moi! Da luu best model (mAP@50-95 = {map_tot_nhat:.4f})")
    else:
        so_epoch_khong_cai_thien += 1
        print(f"  -- Khong cai thien ({so_epoch_khong_cai_thien}/{PATIENCE})")

    ghi_nhat_ky()
    lr_scheduler.step()

    if so_epoch_khong_cai_thien >= PATIENCE:
        print(f"\nEARLY STOPPING: da {PATIENCE} epoch lien tiep khong cai thien.")
        print(f"Dung o epoch {epoch}. Model tot nhat la epoch {epoch_tot_nhat}.")
        break

ket_qua_cuoi = ghi_nhat_ky()
print(f"\nHoan tat. Chay {len(lich_su)}/{SO_EPOCHS} epochs.")
print(f"Best model: epoch {epoch_tot_nhat}, mAP@50-95 = {map_tot_nhat:.4f}")
print(f"Trong so : {DUONG_DAN_BEST}")
print(f"Nhat ky  : {DUONG_DAN_JSON}")

## Cell 8: Ve Learning Curve

In [ ]:
import matplotlib.pyplot as plt


def ve_learning_curve(ket_qua_lich_su, duong_dan_luu_anh):
    """Ve 2 do thi canh nhau: duong cong Loss va duong cong mAP theo tung epoch."""
    lich_su = ket_qua_lich_su['history']
    cac_epoch = [m['epoch_id'] for m in lich_su]

    fig, (truc_trai, truc_phai) = plt.subplots(1, 2, figsize=(14, 5))

    # Do thi 1: Loss - dung de phat hien Overfitting (val_loss quay dau di len)
    truc_trai.plot(cac_epoch, [m['train_loss'] for m in lich_su], label='Train Loss')
    truc_trai.plot(cac_epoch, [m['val_loss'] for m in lich_su], label='Val Loss')
    truc_trai.set_xlabel('Epoch')
    truc_trai.set_ylabel('Loss')
    truc_trai.set_title(f"Learning Curve - {ket_qua_lich_su['model']}")
    truc_trai.legend()
    truc_trai.grid(alpha=0.3)

    # Do thi 2: mAP - dung de xac nhan diem hoi tu that su
    truc_phai.plot(cac_epoch, [m['mAP_50'] for m in lich_su], label='mAP@50')
    truc_phai.plot(cac_epoch, [m['mAP_50_95'] for m in lich_su], label='mAP@50-95')
    truc_phai.set_xlabel('Epoch')
    truc_phai.set_ylabel('mAP')
    truc_phai.set_title('Duong cong do chinh xac')
    truc_phai.legend()
    truc_phai.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(duong_dan_luu_anh, dpi=130)
    plt.show()
    print(f"Da luu bieu do: {duong_dan_luu_anh}")

In [ ]:
lich_su_frcnn = json.load(open(DUONG_DAN_JSON, encoding='utf-8'))
ve_learning_curve(lich_su_frcnn, '/kaggle/working/faster_rcnn_learning_curve.png')

## Cell 9: Nen ket qua de tai ve

In [ ]:
!cp /kaggle/working/faster_rcnn_training_history.json /kaggle/working/faster_rcnn_v3/
!cp /kaggle/working/faster_rcnn_learning_curve.png /kaggle/working/faster_rcnn_v3/
!zip -r -q /kaggle/working/faster_rcnn_v3_results.zip /kaggle/working/faster_rcnn_v3
print("Da nen xong. Tai file faster_rcnn_v3_results.zip o cot Output ben phai.")